# LAB·J2 · The recompile hunt

**Hardware:** any machine. ~1.5 h.

`jit` keeps a cache of compiled executables, keyed on a few concrete things: which function you called, the pytree structure of its arguments, every leaf's shape and dtype, and the value of anything marked static. Get that key right in your head and a recompile stops being weather. It becomes arithmetic you can do before you run anything.

Predict before you run, in every cell that asks for it.


In [ ]:
import jax
import jax.numpy as jnp

jax.config.update("jax_log_compiles", True)
print(jax.__version__, jax.devices())


## Three calls, one cache

Here is a jitted function with one static argument, `k`. Read the three calls below and predict, for each one, whether it triggers a fresh trace and compile or reuses an executable already sitting in the cache.

```python
from functools import partial

import jax
import jax.numpy as jnp

@partial(jax.jit, static_argnames="k")
def topk_sum(x, k):
    return jax.lax.top_k(x, k)[0].sum()

x = jnp.arange(100.0)
topk_sum(x, 3)   # call 1
topk_sum(x, 3)   # call 2
topk_sum(x, 5)   # call 3
```


**Your prediction:**


In [ ]:
from functools import partial

import jax
import jax.numpy as jnp

@partial(jax.jit, static_argnames="k")
def topk_sum(x, k):
    return jax.lax.top_k(x, k)[0].sum()

x = jnp.arange(100.0)
topk_sum(x, 3)   # trace + compile, k = 3 baked in
topk_sum(x, 3)   # cache hit
topk_sum(x, 5)   # new static value: trace + compile again


## Reading the log

With `jax_log_compiles` on, the first call to `topk_sum(x, 3)` logs a trace and compile: nothing in the cache matches this function, this argument structure, `float32[100]`, and `k = 3` yet. The second call with identical arguments hits the cache silently; no log line, because every part of the key is unchanged. The third call changes one component, the static value of `k`, from 3 to 5, and that alone forces a new trace and compile, even though `x` never changed at all.

That is the whole mental model for jit's cache: not "sometimes it recompiles," but "here is exactly which of four things changed, and here is what changed it."


## The churn you cause without meaning to

Static arguments buy you specialization: `k` gets baked into the compiled program as a constant, so `lax.top_k` can be sized exactly for it. The cost is that every distinct value of a static argument is a distinct cache entry. Call `topk_sum` with ten different values of `k` and you get ten compiles, not one function handling all ten.

Predict how many trace-and-compile events the loop below logs.

```python
for k in range(1, 11):
    topk_sum(x, k)
```


**Your prediction:**


In [ ]:
for k in range(1, 11):
    topk_sum(x, k)


## Ten values, ten compiles

Every iteration changes the static value of `k`, so every iteration is a cache miss: ten trace-and-compile events from one Python loop. This is the classic churn bug the cache key predicts directly. A high-cardinality argument, something like a per-call size or a per-call id, marked static turns what looks like one function into as many compiled programs as values you ever pass it.

Call `topk_sum(x, 3)` once more below. That value was compiled in the first section, so this call should be silent: no new log line, an actual cache hit.


In [ ]:
topk_sum(x, 3)   # already compiled earlier: silent


## Driving it to silence

A steady-state training loop should look exactly like that last call: silent. Every step after warmup hits the cache, because the shapes, dtypes, and static values repeat every step. `jax_log_compiles` is the tool that proves it. Turn it on, run your real loop, and if compile log lines keep appearing past the first few steps, one of the four key components is still drifting: function identity, pytree structure, shape and dtype, or a static value. The recompile hunt is finding which one and stopping it from changing.


## Compiling ahead of time

`jit` normally traces and compiles the first time you call a function. You can also do both steps by hand, before any call, and look at what came out. `lower` gives you the StableHLO the compiler will work from; `compile` turns that into an executable; `cost_analysis` asks the compiler what it thinks that executable costs.

Predict the FLOP count `cost_analysis` reports for a plain `256 x 256` matmul, `x @ x`, before you run the cell.


**Your prediction:**


In [ ]:
import jax
import jax.numpy as jnp

lowered = jax.jit(lambda x: x @ x).lower(jnp.ones((256, 256)))
print(lowered.as_text()[:300])   # StableHLO, before anything runs
compiled = lowered.compile()
print(compiled.cost_analysis()["flops"])   # the compiler's own estimate: 33554432.0


## Where the number comes from

`cost_analysis()["flops"]` reports 33554432.0, and that is not a rough guess: it is exactly $2 \times 256^3$, the FLOP count of one matmul at that size, one multiply and one add per output element summed over the contraction. Nothing ran to produce this number. `lower` and `compile` did their work, and `cost_analysis` read it straight off the compiled program.

On jax versions before 0.5, `cost_analysis()` returned this same dictionary wrapped in a one-element list; the dict form used here is what 0.4.38 returns directly. Reading a cost before running anything catches an accidentally quadratic reshape, or a matmul at the wrong shape, before you sit through it running.


## Mark it run

You have read a cache key off real log lines, caused ten compiles on purpose and then stopped them, and pulled a FLOP count out of a compiled program before it ever executed. Go back to chapter 3 on the chapter page and tick LAB·J2 as run.
